---
tags: [usage, resource-estimation, primitive]
---

# Block-Encoding Resource Estimates

Block-encoding is the interface between a Hamiltonian representation and
qubitized FTQC algorithms. This notebook shows how to model PREPARE, SELECT,
reflection, and QPE readout costs separately before committing to any backend
circuit decomposition.

In [1]:
# Install the latest Qamomile through pip!
# !pip install qamomile

In [2]:
import sympy as sp

import qamomile.observable as qm_o
from qamomile.circuit.estimator.algorithmic import (
    BlockEncodingResource,
    ChemistryQPEMethod,
    ChemistryQPEModel,
    FTQCCostModel,
    FTQCResourceQuantity,
    block_encoding_from_chemistry_model,
    compare_ftqc_resource_estimates,
    estimate_qubitized_qpe_from_block_encoding,
    summarize_pauli_hamiltonian,
)

## Workflow

A qubitized walk is usually built from reusable subroutines:

- PREPARE loads amplitudes or coefficients into an index register.
- SELECT applies the indexed unitary or oracle.
- A reflection completes the walk operator.

Qamomile keeps these as resource-model fields. The circuit IR does not need a
special block-encoding operation just to compare algorithmic costs.

## Minimal Example

The numbers below are synthetic. They show the accounting pattern:
one qubitized walk costs two PREPARE calls, one SELECT call, and one
reflection.

In [3]:
block = BlockEncodingResource(
    system_qubits=12,
    normalization=sp.Integer(240),
    prepare_cost_toffoli=30,
    select_cost_toffoli=120,
    reflection_cost_toffoli=8,
    ancilla_qubits=5,
    name="toy_lcu",
)

print(block.to_dict())

assert block.logical_qubits == 17
assert block.walk_cost_toffoli == 188
assert block.resource_values()[FTQCResourceQuantity.WALK_COST_TOFFOLI] == 188
assert block.resource_values()[FTQCResourceQuantity.PREPARE_COST_TOFFOLI] == 30
assert block.resource_values()[FTQCResourceQuantity.SELECT_COST_TOFFOLI] == 120
assert block.resource_values()[FTQCResourceQuantity.REFLECTION_COST_TOFFOLI] == 8

{'name': 'toy_lcu', 'system_qubits': '12', 'normalization': '240', 'select_cost_toffoli': '120', 'prepare_cost_toffoli': '30', 'reflection_cost_toffoli': '8', 'ancilla_qubits': '5', 'logical_qubits': '17', 'walk_cost_toffoli': '188', 'references': []}


These subroutine costs also use canonical quantity keys. That makes reports
less dependent on the field names of `BlockEncodingResource` itself.

In [4]:
for quantity in (
    FTQCResourceQuantity.SYSTEM_QUBITS,
    FTQCResourceQuantity.BLOCK_ENCODING_ANCILLA_QUBITS,
    FTQCResourceQuantity.PREPARE_COST_TOFFOLI,
    FTQCResourceQuantity.SELECT_COST_TOFFOLI,
    FTQCResourceQuantity.REFLECTION_COST_TOFFOLI,
    FTQCResourceQuantity.WALK_COST_TOFFOLI,
):
    print(quantity.value, "=", block.resource_values()[quantity])

system_qubits = 12
block_encoding_ancilla_qubits = 5
prepare_cost_toffoli = 30
select_cost_toffoli = 120
reflection_cost_toffoli = 8
walk_cost_toffoli = 188


## Qubitized QPE

QPE repeatedly calls the qubitized walk. For energy precision
$\epsilon$, the symbolic call proxy is $\alpha / \epsilon$, where
$\alpha$ is the block-encoding normalization.

In [5]:
architecture = FTQCCostModel(
    physical_qubits_per_logical=100,
    logical_cycle_time_seconds=sp.Float("1e-6"),
    factory_qubits=2000,
    toffoli_throughput_per_second=sp.Float("5e5"),
)

estimate = estimate_qubitized_qpe_from_block_encoding(
    block,
    precision=sp.Integer(3),
    qpe_register_qubits=6,
    cost_model=architecture,
)

print("iterations:", estimate.qpe_iterations)
print("Toffoli gates:", estimate.toffoli_gates)
print("logical qubits:", estimate.logical_qubits)

assert estimate.qpe_iterations == 80
assert estimate.toffoli_gates == 15040
assert estimate.logical_qubits == 23
assert estimate.physical_qubits == 4300
assert estimate.assumptions["block_encoding"] == "toy_lcu"
assert estimate.resource_values()[FTQCResourceQuantity.QPE_REGISTER_QUBITS] == 6
assert estimate.to_dict()["algorithm_values"]["prepare_cost_toffoli"] == "30"
assert any(reference.key == "arXiv:1610.06546" for reference in estimate.references)

iterations: 80
Toffoli gates: 15040
logical qubits: 23


## Compare Representations

A new factorization or symmetry shift can reduce the normalization while
increasing SELECT/PREPARE cost. Keeping the fields separate makes that
trade-off visible.

In [6]:
compressed_block = BlockEncodingResource(
    system_qubits=12,
    normalization=sp.Integer(120),
    prepare_cost_toffoli=36,
    select_cost_toffoli=144,
    reflection_cost_toffoli=8,
    ancilla_qubits=7,
    name="compressed_toy_lcu",
)

compressed_estimate = estimate_qubitized_qpe_from_block_encoding(
    compressed_block,
    precision=sp.Integer(3),
    qpe_register_qubits=6,
    cost_model=architecture,
)

comparison = compare_ftqc_resource_estimates(
    estimate,
    compressed_estimate,
    quantities=("qpe_iterations", "toffoli_gates", "logical_qubits"),
)

for row in comparison:
    print(row.label, "ratio:", sp.N(row.ratio, 4))

assert comparison[0].ratio == sp.Rational(1, 2)
assert sp.simplify(comparison[1].ratio - sp.Rational(28, 47)) == 0
assert sp.simplify(comparison[2].ratio - sp.Rational(25, 23)) == 0

QPE iterations ratio: 0.5000
Toffoli gates ratio: 0.5957
Logical qubits ratio: 1.087


## Bridge from Chemistry Models

Chemistry estimators often start from a Hamiltonian summary and a
representation-level walk cost. The bridge below turns that model into the
same block-encoding contract, so reports can compare chemistry-specific and
block-encoding views without duplicating inputs.

In [7]:
toy_chemistry = summarize_pauli_hamiltonian(
    2 * qm_o.Z(0) + 3 * qm_o.X(1),
    n_spin_orbitals=8,
    source="toy_chemistry",
)
chemistry_model = ChemistryQPEModel(
    hamiltonian=toy_chemistry.with_lambda_scale(sp.Rational(1, 2)),
    method=ChemistryQPEMethod.SYMMETRY_COMPRESSED_DF,
    walk_cost_toffoli=100,
    second_factor_rank=4,
    description="compressed chemistry toy",
)

chemistry_block = block_encoding_from_chemistry_model(chemistry_model)
chemistry_estimate = estimate_qubitized_qpe_from_block_encoding(
    chemistry_block,
    precision=1,
)

print(chemistry_block.to_dict())

assert chemistry_model.logical_qubit_count == 16
assert chemistry_block.logical_qubits == 16
assert sp.Abs(chemistry_estimate.qpe_iterations - 2.5) < sp.Float("1e-12")
assert sp.Abs(chemistry_estimate.toffoli_gates - 250) < sp.Float("1e-12")
assert any(
    reference.key == "arXiv:2403.03502" for reference in chemistry_block.references
)

{'name': 'compressed chemistry toy', 'system_qubits': '8', 'normalization': '2.50000000000000', 'select_cost_toffoli': '100', 'prepare_cost_toffoli': '0', 'reflection_cost_toffoli': '0', 'ancilla_qubits': '8', 'logical_qubits': '16', 'walk_cost_toffoli': '100', 'references': [{'key': 'arXiv:1902.02134', 'title': 'Qubitization of Arbitrary Basis Quantum Chemistry Leveraging Sparsity and Low Rank Factorization', 'url': 'https://arxiv.org/abs/1902.02134', 'note': 'Provides sparse and low-rank qubitized chemistry cost models that motivate representation-specific logical-resource scaling.'}, {'key': 'arXiv:2403.03502', 'title': 'Reducing the runtime of fault-tolerant quantum simulations in chemistry through symmetry-compressed double factorization', 'url': 'https://arxiv.org/abs/2403.03502', 'note': 'Introduces symmetry-compressed double factorization and reports Hamiltonian 1-norm and Toffoli-count reductions.'}, {'key': 'arXiv:2412.01338', 'title': 'Simultaneously optimizing symmetry shif

## Notes

:::{note}
Treat `BlockEncodingResource` as a symbolic contract for algorithm design.
It records what the estimator consumes; it does not claim that a particular
PREPARE or SELECT circuit has already been synthesized for a backend.
:::

## Summary

In this notebook, we learned:

- Block-encoding estimates separate normalization, PREPARE, SELECT,
  reflection, ancilla, and QPE readout costs.
- PREPARE, SELECT, reflection, workspace, and QPE readout quantities have
  canonical resource keys for downstream reports.
- Qubitized QPE composes the block-encoding walk cost with
  normalization-over-precision iterations.
- Chemistry QPE models can be converted into the same block-encoding
  contract for cross-view comparisons.
- Representation trade-offs can reduce total Toffoli count even when one
  subroutine becomes more expensive.